In [9]:
from __future__ import annotations

import os

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon
from matplotlib.widgets import Slider, Button, RadioButtons, TextBox
from scipy.ndimage import map_coordinates, uniform_filter1d
import matplotlib.ticker as mticker

%matplotlib qt

# --------------------------------------------------------------------------
# X-ray line energies (keV).  Used when HyperSpy's database is unavailable.
# --------------------------------------------------------------------------
LINE_ENERGIES = {
    "C":  {"Ka": 0.277},
    "N":  {"Ka": 0.392},
    "O":  {"Ka": 0.525},
    "F":  {"Ka": 0.677},
    "Na": {"Ka": 1.041},
    "Mg": {"Ka": 1.254},
    "Al": {"Ka": 1.487},
    "Si": {"Ka": 1.740},
    "P":  {"Ka": 2.013},
    "S":  {"Ka": 2.307},
    "Cl": {"Ka": 2.622},
    "K":  {"Ka": 3.313},
    "Ca": {"Ka": 3.691},
    "Ti": {"Ka": 4.510, "La": 0.452},
    "V":  {"Ka": 4.952, "La": 0.511},
    "Cr": {"Ka": 5.414, "La": 0.573},
    "Mn": {"Ka": 5.898, "La": 0.637},
    "Fe": {"Ka": 6.403, "La": 0.705},
    "Co": {"Ka": 6.930, "La": 0.776},
    "Ni": {"Ka": 7.477, "La": 0.851},
    "Cu": {"Ka": 8.047, "La": 0.930},
    "Zn": {"Ka": 8.638, "La": 1.012},
    "Ga": {"Ka": 9.251, "La": 1.098},
    "Ge": {"Ka": 9.885, "La": 1.188},
    "Mo": {"Ka": 17.478, "La": 2.293},
    "Ag": {"Ka": 22.162, "La": 2.984},
    "Sn": {"La": 3.444},
    "Ta": {"La": 8.146, "Ma": 1.710},
    "W":  {"La": 8.396, "Ma": 1.775},
    "Pt": {"La": 9.442, "Ma": 2.048},
    "Au": {"La": 9.712, "Ma": 2.120},
    "Pb": {"La": 10.550, "Ma": 2.342},
    "Bi": {"La": 10.839, "Ma": 2.419, "Mb": 2.526},
}

ELEMENT_COLORS = {
    "C": "#000000", "O": "#1f4fff", "Ni": "#66dd11", "Fe": "#ff7f0e", 
    "Cr": "#9467bd", "Al": "#17becf", "Ti": "#e377c2", "Mn": "#8c564b", 
    "N": "#7f7f7f", "Co": "#c8a05a", "Au": "#d62728", "Bi": "#2ca02c",
}
FALLBACK_COLORS = ["#d62728", "#9467bd", "#8c564b", "#e377c2",
                   "#7f7f7f", "#bcbd22", "#17becf", "#ff7f0e"]


# --------------------------------------------------------------------------
# Line energy lookup
# --------------------------------------------------------------------------
def get_line_energy(element, line=None, e_max=None):
    """Return (line_name, energy_keV) for `element`, preferring K then L then M.

    If `e_max` is given, the highest-energy line that still fits in the
    spectrum range is chosen (so Cu-K is skipped for a 5 keV spectrum).
    """
    table = None
    try:  # HyperSpy's database is more complete if it is installed
        import hyperspy.api as hs
        props = hs.material.elements[element]["Atomic_properties"]["Xray_lines"]
        table = {k: float(v["energy (keV)"]) for k, v in props.items()}
    except Exception:
        table = dict(LINE_ENERGIES.get(element, {}))

    if not table:
        raise ValueError(f"No X-ray line data for element '{element}'. "
                         f"Add it to LINE_ENERGIES or install hyperspy.")

    if line is not None:
        if line not in table:
            raise ValueError(f"Line '{line}' not tabulated for {element}. "
                             f"Available: {sorted(table)}")
        return line, table[line]

    for fam in ("Ma", "La", "Ka"):
        cands = {k: v for k, v in table.items() if k.startswith(fam[0])}
        if fam in table:
            e = table[fam]
        elif cands:
            fam, e = sorted(cands.items(), key=lambda kv: -kv[1])[0]
        else:
            continue
        if e_max is None or e < 0.95 * e_max:
            return fam, e

    fam, e = min(table.items(), key=lambda kv: kv[1])
    return fam, e


# --------------------------------------------------------------------------
# Elemental map from the SI by window integration + Rolling low-percentile background
# --------------------------------------------------------------------------
def rolling_background(spectrum, window_frac=0.05, percentile=5, smooth=True,
                       fast=True, poisson_correct=True, stride=None):
    """Estimate a background as a rolling low percentile along the last axis.
 
    Works on a single spectrum (nE,) or a whole SI (ny, nx, nE).  No window
    placement is needed: the percentile floor follows the bremsstrahlung and
    steps over the peaks, because within any window most channels are
    background.
 
    window_frac : window half-width as a fraction of the number of channels
    percentile  : which percentile is taken as the floor
    smooth      : moving-average the result to remove blocky steps
    fast        : evaluate the percentile on a decimated channel grid and
        interpolate.  ~4x faster than the exact filter and agrees with it to
        well under a percent after smoothing.  Set False for the exact result.
    poisson_correct : a low percentile of Poisson noise sits below the true
        mean by roughly |z|*sqrt(lambda), which leaves a positive pedestal
        under every channel.  This inverts that bias.
    """
    spec = np.asarray(spectrum, float)
    nE = spec.shape[-1]
    win = max(int(nE * window_frac), 10)
 
    if fast:
        stride = stride or max(win // 4, 1)
        centres = np.arange(0, nE, stride)
        if centres[-1] != nE - 1:
            centres = np.append(centres, nE - 1)
        coarse = np.empty(spec.shape[:-1] + (len(centres),))
        for j, c in enumerate(centres):
            coarse[..., j] = np.percentile(
                spec[..., max(0, c - win):min(nE, c + win)], percentile, axis=-1)
        xs = np.arange(nE)
        idx = np.clip(np.searchsorted(centres, xs, side="right") - 1,
                      0, len(centres) - 2)
        x0, x1 = centres[idx], centres[idx + 1]
        w = (xs - x0) / (x1 - x0)
        bkgd = coarse[..., idx] * (1 - w) + coarse[..., idx + 1] * w
    else:
        from scipy.ndimage import percentile_filter
        size = tuple([1] * (spec.ndim - 1)) + (2 * win + 1,)
        bkgd = percentile_filter(spec, percentile=percentile, size=size,
                                 mode="nearest")
 
    if smooth:
        bkgd = uniform_filter1d(bkgd, size=win, axis=-1, mode="nearest")
 
    if poisson_correct:
        # observed = lambda + z*sqrt(lambda)  ->  solve for lambda
        try:
            from scipy.stats import norm
            z = float(norm.ppf(percentile / 100.0))
        except Exception:
            z = -1.6449
        b = np.clip(bkgd, 0, None)
        bkgd = ((-z + np.sqrt(z * z + 4 * b)) / 2.0) ** 2
 
    return bkgd
 
 
def _family_lines(element, fam_letter):
    """All tabulated lines of one family (e.g. all M lines) for an element."""
    try:
        import hyperspy.api as hs
        props = hs.material.elements[element]["Atomic_properties"]["Xray_lines"]
        table = {k: float(v["energy (keV)"]) for k, v in props.items()}
    except Exception:
        table = dict(LINE_ENERGIES.get(element, {}))
    return sorted(v for k, v in table.items() if k.startswith(fam_letter))
 
 
def element_map(si, energy, element, line=None, width_kev=0.16,
                span_family=False, verbose=False):
    """Integrate the SI over a window around an X-ray line.
 
    This is pure window integration - no background handling.  Strip the
    background from the SI first with `rolling_background`, which `launch`
    does for you.
 
    Parameters
    ----------
    si : (ny, nx, nE) array, ideally already background-subtracted
    energy : (nE,) array of channel energies in keV
    width_kev : full width of the signal window
    span_family : widen the window to cover every line of the same family,
        e.g. Bi-Ma and Bi-Mb together.  Worth setting for M and L lines,
        where splitting the family throws away a large share of the counts.
 
    Returns (map, line_name, line_energy, (window_lo, window_hi)).
    """
    line_name, e0 = get_line_energy(element, line, e_max=float(energy[-1]))
 
    lo_e, hi_e = e0 - width_kev / 2, e0 + width_kev / 2
    if span_family:
        fam = [f for f in _family_lines(element, line_name[0]) if abs(f - e0) < 0.6]
        if fam:
            lo_e = min(lo_e, min(fam) - width_kev / 2)
            hi_e = max(hi_e, max(fam) + width_kev / 2)
 
    sig = (energy >= lo_e) & (energy <= hi_e)
    if sig.sum() == 0:
        raise ValueError(f"{element}-{line_name} ({e0:.3f} keV) lies outside "
                         f"the spectrum range {energy[0]:.2f}-{energy[-1]:.2f} keV.")
 
    m = si[:, :, sig].sum(axis=2).astype(float)
    win = (float(energy[sig][0]), float(energy[sig][-1]))
 
    if verbose:
        print(f"    {element}-{line_name}: window {win[0]:.3f}-{win[1]:.3f} keV "
              f"({int(sig.sum())} ch), max {m.max():.0f}")
 
    return m, line_name, e0, win


# --------------------------------------------------------------------------
# Core profile extraction (this is the piece you would reuse elsewhere)
# --------------------------------------------------------------------------
def line_profile(img, p0, p1, width=1.0, n_along=None, n_across=None, order=1):
    """Average `img` over a rectangular band from p0 to p1.

    p0, p1 : (x, y) in pixel coordinates (column, row)
    width  : band width in pixels, perpendicular to the line
    n_along: number of samples along the line (default ~1 per pixel)
    n_across: number of perpendicular samples (default ~1 per pixel of width)
    order  : spline order for map_coordinates (1 = bilinear, 0 = nearest)

    Returns (t, profile) with t in pixels measured from p0.
    """
    x0, y0 = float(p0[0]), float(p0[1])
    x1, y1 = float(p1[0]), float(p1[1])
    length = np.hypot(x1 - x0, y1 - y0)
    if length < 1e-9:
        return np.zeros(1), np.array([float(img[int(round(y0)), int(round(x0))])])

    ux, uy = (x1 - x0) / length, (y1 - y0) / length      # unit vector along
    px, py = -uy, ux                                     # unit vector across

    if n_along is None:
        n_along = max(int(round(length)) + 1, 2)
    if n_across is None:
        n_across = max(int(round(width)), 1)

    t = np.linspace(0.0, length, n_along)
    s = np.array([0.0]) if n_across == 1 else np.linspace(-width / 2, width / 2, n_across)

    X = x0 + ux * t[None, :] + px * s[:, None]
    Y = y0 + uy * t[None, :] + py * s[:, None]

    band = map_coordinates(np.asarray(img, dtype=float), [Y, X],
                           order=order, mode="nearest")
    return t, band.mean(axis=0)


def band_mask(shape, p0, p1, width):
    """Boolean mask of the pixels lying inside the integration band."""
    ny, nx = shape
    x0, y0 = float(p0[0]), float(p0[1])
    x1, y1 = float(p1[0]), float(p1[1])
    length = np.hypot(x1 - x0, y1 - y0)
    if length < 1e-9:
        return np.zeros(shape, bool)
    ux, uy = (x1 - x0) / length, (y1 - y0) / length
    px, py = -uy, ux
    yy, xx = np.mgrid[0:ny, 0:nx]
    dx, dy = xx - x0, yy - y0
    along = dx * ux + dy * uy
    across = np.abs(dx * px + dy * py)
    return (along >= 0) & (along <= length) & (across <= max(width / 2, 0.5))


class LineScanTool:
    HANDLE_PIX = 10          # grab radius in display pixels
 
    def __init__(self, adf, maps, si=None, energy=None, pixel_size=1.0,
                 units="px", title="EDS line scan", line_info=None,
                 init_width=5.0, si_bg=None, panel_pad=0.09,
                 panel_shares=(0.46, 0.30, 0.24),
                 panel_rect=(0.075, 0.105, 0.545, 0.950),
                 panel_widths=(1.0, 0.7, 0.7),
                 col_x=0.775, col_w=0.195, figsize=(10.0, 10.0)):
        self.adf = np.asarray(adf, float)
        self.maps = maps                       # dict {label: 2D array}
        self.labels = list(maps.keys())
        self.si = si
        self.si_bg = si_bg
        self.panel_pad = float(panel_pad)
        self.panel_shares = tuple(panel_shares)
        self.panel_rect = tuple(panel_rect)
        self.panel_widths = tuple(panel_widths)
        self.col_x = float(col_x)
        self.col_w = float(col_w)
        self.figsize = tuple(figsize)
        self._buttons = []
        self.energy = energy
        self.pixel_size = float(pixel_size)
        self.units = units
        self.line_info = line_info or {}
        self.ny, self.nx = self.adf.shape
 
        self.norm_mode = "raw"
        self.log_scale = False
        self.smooth = 1
        self.active = None                     # 'p0' | 'p1' | 'mid' | None
        self.drag_new = False
 
        # default line: horizontal, through the middle
        self.p0 = np.array([0.10 * self.nx, 0.50 * self.ny])
        self.p1 = np.array([0.90 * self.nx, 0.50 * self.ny])
        self.p0_default, self.p1_default = self.p0.copy(), self.p1.copy()
        self.width = float(init_width)
        self.n_samples = 0                     # 0 -> auto (1 per pixel)
 
        self._build_figure(title)
        self.update()
 
    # ---------------------------------------------------------------- colors
    def color(self, label):
        el = label.split("-")[0].split("_")[0]
        if el in ELEMENT_COLORS:
            return ELEMENT_COLORS[el]
        return FALLBACK_COLORS[self.labels.index(label) % len(FALLBACK_COLORS)]
 
    # ---------------------------------------------------------------- layout
    def _panel_axes(self, has_spec):
        """Stack the left-hand panels top-down with `panel_pad` between them.
 
        panel_rect   : (left, bottom, width, top) of the whole stack
        panel_shares : fraction of the remaining height for image / profile /
            spectrum.  Only the first two are used when there is no spectrum.
        panel_pad    : vertical gap between panels, in figure fractions.
            Raise it if the x tick labels of one panel collide with the next.
        """
        left, bottom, width, top = self.panel_rect
        shares = list(self.panel_shares[:3 if has_spec else 2])
        shares = [s / sum(shares) for s in shares]
        n = len(shares)
        avail = (top - bottom) - self.panel_pad * (n - 1)
        heights = [s * avail for s in shares]
 
        axes, y = [], top
        for h in heights:
            y -= h
            axes.append(self.fig.add_axes([left, y, width, h]))
            y -= self.panel_pad
 
        self.ax_img, self.ax_prof = axes[0], axes[1]
        self.ax_spec = axes[2] if has_spec else None
        return axes
 
    def set_panel_pad(self, pad=None, shares=None):
        """Re-space the panels after the window is already open."""
        if pad is not None:
            self.panel_pad = float(pad)
        if shares is not None:
            self.panel_shares = tuple(shares)
        left, bottom, width, top = self.panel_rect
        has_spec = self.ax_spec is not None
        s = list(self.panel_shares[:3 if has_spec else 2])
        s = [v / sum(s) for v in s]
        avail = (top - bottom) - self.panel_pad * (len(s) - 1)
        y = top
        for ax, frac in zip([self.ax_img, self.ax_prof, self.ax_spec], s):
            h = frac * avail
            y -= h
            ax.set_position([left, y, width, h])
            y -= self.panel_pad
        self.fig.canvas.draw_idle()
 
    def _build_figure(self, title):
        self.fig = plt.figure(figsize=self.figsize)
        self.fig.canvas.manager.set_window_title(title)
 
        has_spec = self.si is not None and self.energy is not None
        for ax in self._panel_axes(has_spec):
            pass
        if not has_spec:
            self.ax_spec = None
 
        # ---- survey image
        self.im = self.ax_img.imshow(self.adf, cmap="gray", origin="upper",
                                     interpolation="nearest")
        self.ax_img.set_title(title, fontsize=13, fontweight="bold",
                              color="tab:blue", pad=10)
        self.ax_img.set_xlabel("x [px]", labelpad=2)
        self.ax_img.set_ylabel("y [px]", labelpad=2)
        self.ax_img.tick_params(labelsize=9)
 
        self.band_patch = Polygon(np.zeros((4, 2)), closed=True, facecolor="red",
                                  alpha=0.18, edgecolor="red", lw=0.8, zorder=3)
        self.ax_img.add_patch(self.band_patch)
        self.line_artist = Line2D([], [], color="red", lw=2.0, zorder=4)
        self.ax_img.add_line(self.line_artist)
        self.h_ends = Line2D([], [], ls="none", marker="s", ms=8, mfc="yellow",
                             mec="k", zorder=5)
        self.h_mid = Line2D([], [], ls="none", marker="o", ms=7, mfc="white",
                            mec="red", zorder=5)
        self.ax_img.add_line(self.h_ends)
        self.ax_img.add_line(self.h_mid)
 
        # ---- profile axes
        self.ax_prof.set_xlabel(f"Distance ({self.units})")
        self.ax_prof.set_ylabel("Intensity [counts]")
        self.ax_prof.grid(alpha=0.3)
        self.prof_lines = {}
        for lab in self.labels:
            (ln,) = self.ax_prof.plot([], [], lw=1.4, color=self.color(lab),
                                      label=self._legend_label(lab))
            self.prof_lines[lab] = ln
        self.ax_prof.legend(loc="upper right", frameon=False, fontsize=9.5, labelspacing=0.9,
                            handlelength=1.6, borderaxespad=0.0)
        self.ax_prof.tick_params(labelsize=9)
 
        # ---- spectrum axes
        if self.ax_spec is not None:
            self.ax_spec.set_xlabel("Energy (keV)", labelpad=2)
            self.ax_spec.set_ylabel("Counts", labelpad=2)
            self.ax_spec.tick_params(labelsize=9)
            self.ax_spec.grid(alpha=0.3)
            (self.spec_line,) = self.ax_spec.plot([], [], lw=0.9, color="k")
            (self.bg_line,) = self.ax_spec.plot([], [], lw=1.0, color="0.55",
                                                ls="--", label="background")
            self.spec_label = self.ax_spec.text(
                0.99, 0.93, "", transform=self.ax_spec.transAxes, ha="right",
                va="top", fontsize=9, color="0.35")
            self.spec_marks = []
            for lab in self.labels:
                e0 = self.line_info.get(lab, {}).get("energy")
                if e0 is None:
                    continue
                v = self.ax_spec.axvline(e0, color=self.color(lab), lw=1.0,
                                         ls="--", alpha=0.9)
                self.spec_marks.append(v)
                w = self.line_info[lab].get("window")
                if w:
                    self.ax_spec.axvspan(w[0], w[1], color=self.color(lab),
                                         alpha=0.12, lw=0)
 
        # ---- widgets ------------------------------------------------------
        # Laid out top-down from a cursor so nothing can collide.  Adjust the
        # column with COL_X / COL_W, or the vertical rhythm with GAP.
        X, W = self.col_x, self.col_w
        GAP, SLIDER_H, BTN_H, TB_H = 0.026, 0.020, 0.038, 0.030
        y = 0.955
 
        self.ax_w = self.fig.add_axes([X, y, W, SLIDER_H])
        self.s_width = Slider(self.ax_w, "width [px]", 1, max(4, self.nx // 3),
                              valinit=self.width, valstep=1)
        self.s_width.on_changed(lambda v: self.update())
        y -= SLIDER_H + GAP
 
        self.ax_n = self.fig.add_axes([X, y, W, SLIDER_H])
        nmax = int(2 * np.hypot(self.nx, self.ny))
        self.s_nsamp = Slider(self.ax_n, "samples", 0, nmax, valinit=0, valstep=1)
        self.s_nsamp.on_changed(lambda v: self.update())
        y -= SLIDER_H + GAP
 
        self.ax_s = self.fig.add_axes([X, y, W, SLIDER_H])
        self.s_smooth = Slider(self.ax_s, "smooth", 1, 51, valinit=1, valstep=2)
        self.s_smooth.on_changed(lambda v: self.update())
        y -= SLIDER_H + 2.4 * GAP          # extra room for the radio title
 
        RADIO_H = 0.072
        y -= RADIO_H
        self.ax_norm = self.fig.add_axes([X, y, W, RADIO_H])
        self.ax_norm.set_title("y values", fontsize=9, loc="left", pad=4)
        self.r_norm = RadioButtons(self.ax_norm, ("raw counts", "norm %"), active=0)
        self.r_norm.on_clicked(self._on_norm)
        y -= 2.4 * GAP
 
        y -= RADIO_H
        self.ax_scale = self.fig.add_axes([X, y, W, RADIO_H])
        self.ax_scale.set_title("y scale", fontsize=9, loc="left", pad=4)
        self.r_scale = RadioButtons(self.ax_scale, ("linear", "log"), active=0)
        self.r_scale.on_clicked(self._on_scale)
        y -= 2.6 * GAP
 
        # numeric coordinate entry, two per row
        self.fig.text(X, y + 0.012, "line endpoints [px]", fontsize=9,
                      va="bottom", color="0.25")
        self.tb_axes = {}
        tb_w = (W - 0.055) / 2
        for row, pair in enumerate((("x0", "y0"), ("x1", "y1"))):
            y -= TB_H + 0.012
            for col, name in enumerate(pair):
                ax = self.fig.add_axes([X + 0.030 + col * (tb_w + 0.028),
                                        y, tb_w, TB_H])
                tb = TextBox(ax, name + " ", initial="0")
                tb.on_submit(self._on_coord_submit)
                self.tb_axes[name] = tb
        y -= 1.9 * GAP
 
        for label, cb in (("apply coordinates", lambda e: self._apply_coords()),
                          ("reset line", lambda e: self.reset()),
                          ("save CSV", lambda e: self.save_csv()),
                          ("save PNG", lambda e: self.save_png())):
            y -= BTN_H
            ax = self.fig.add_axes([X, y, W, BTN_H])
            b = Button(ax, label)
            b.on_clicked(cb)
            self._buttons.append(b)          # keep references alive
            y -= 0.014
        self.b_apply, self.b_reset, self.b_csv, self.b_png = self._buttons
 
        y -= GAP
        self.txt_info = self.fig.text(X, y, "", fontsize=8.5, va="top",
                                      family="monospace", color="0.15")
 
        # help text sits across the bottom, clear of the panels and the column
        self.fig.text(self.panel_rect[0], 0.015,
                      "drag square = endpoint   |   drag circle = whole line   |   shift+drag = new line",
                      fontsize=9, va="bottom", color="0.45", linespacing=1.6)
 
        # ---- events
        c = self.fig.canvas
        c.mpl_connect("button_press_event", self.on_press)
        c.mpl_connect("motion_notify_event", self.on_motion)
        c.mpl_connect("button_release_event", self.on_release)
        c.mpl_connect("key_press_event", self.on_key)
 
    def _legend_label(self, lab):
        info = self.line_info.get(lab, {})
        if "energy" in info:
            return f"{lab}  ({info['energy']:.2f} keV)"
        return lab

    # ---------------------------------------------------------------- events
    def _on_norm(self, label):
        self.norm_mode = "norm" if label.startswith("norm") else "raw"
        self.update()

    def _on_scale(self, label):
        self.log_scale = (label == "log")
        self.update()

    def toggle_log(self):
        """Flip the y axis between linear and log (also updates the radio)."""
        self.r_scale.set_active(0 if self.log_scale else 1)

    def _on_coord_submit(self, _text):
        pass  # applied via the button, so partial typing does not redraw

    def _apply_coords(self):
        try:
            x0 = float(self.tb_axes["x0"].text)
            y0 = float(self.tb_axes["y0"].text)
            x1 = float(self.tb_axes["x1"].text)
            y1 = float(self.tb_axes["y1"].text)
        except ValueError:
            return
        self.p0 = np.clip([x0, y0], [0, 0], [self.nx - 1, self.ny - 1])
        self.p1 = np.clip([x1, y1], [0, 0], [self.nx - 1, self.ny - 1])
        self.update()

    def _near(self, event, pt):
        xd, yd = self.ax_img.transData.transform(pt)
        return np.hypot(event.x - xd, event.y - yd) < self.HANDLE_PIX

    def on_press(self, event):
        if event.inaxes is not self.ax_img or event.button != 1:
            return
        if event.key == "shift":
            self.p0 = np.array([event.xdata, event.ydata])
            self.p1 = self.p0.copy()
            self.active, self.drag_new = "p1", True
            return
        mid = 0.5 * (self.p0 + self.p1)
        if self._near(event, self.p0):
            self.active = "p0"
        elif self._near(event, self.p1):
            self.active = "p1"
        elif self._near(event, mid):
            self.active = "mid"
            self._grab = np.array([event.xdata, event.ydata])

    def on_motion(self, event):
        if self.active is None or event.inaxes is not self.ax_img:
            return
        pos = np.clip([event.xdata, event.ydata], [0, 0], [self.nx - 1, self.ny - 1])
        if self.active == "p0":
            self.p0 = pos
        elif self.active == "p1":
            self.p1 = pos
        elif self.active == "mid":
            d = pos - self._grab
            self.p0, self.p1 = self.p0 + d, self.p1 + d
            self._grab = pos
        self.update()

    def on_release(self, event):
        self.active, self.drag_new = None, False

    def on_key(self, event):
        step = 5.0 if event.key and "shift" in str(event.key) else 1.0
        k = event.key
        if k == "r":
            self.reset(); return
        if k == "l":
            self.toggle_log(); return
        if k == "[":
            self.s_width.set_val(max(1, self.s_width.val - 1)); return
        if k == "]":
            self.s_width.set_val(min(self.s_width.valmax, self.s_width.val + 1)); return
        deltas = {"left": (-step, 0), "right": (step, 0),
                  "up": (0, -step), "down": (0, step)}
        if k in deltas:
            d = np.array(deltas[k])
            self.p0 = self.p0 + d
            self.p1 = self.p1 + d
            self.update()

    def reset(self):
        self.p0, self.p1 = self.p0_default.copy(), self.p1_default.copy()
        self.update()

    # ---------------------------------------------------------------- update
    def compute(self):
        self.width = float(self.s_width.val)
        n = int(self.s_nsamp.val) or None
        self.smooth = int(self.s_smooth.val)

        profs = {}
        t = None
        for lab, m in self.maps.items():
            t, y = line_profile(m, self.p0, self.p1, self.width, n_along=n)
            if self.smooth > 1:
                y = uniform_filter1d(y, self.smooth, mode="nearest")
            profs[lab] = y
        _, adf_p = line_profile(self.adf, self.p0, self.p1, self.width, n_along=n)

        if self.norm_mode == "norm":
            tot = np.sum([np.clip(v, 0, None) for v in profs.values()], axis=0)
            tot[tot <= 0] = np.nan
            profs = {k: 100.0 * np.clip(v, 0, None) / tot for k, v in profs.items()}

        return t, profs, adf_p

    def update(self):
        t_px, profs, _ = self.compute()
        d = t_px * self.pixel_size

        # --- overlay geometry
        v = self.p1 - self.p0
        L = np.hypot(*v)
        if L > 1e-9:
            u = v / L
            p = np.array([-u[1], u[0]]) * self.width / 2
            corners = np.array([self.p0 + p, self.p1 + p, self.p1 - p, self.p0 - p])
            self.band_patch.set_xy(corners)
        self.line_artist.set_data([self.p0[0], self.p1[0]], [self.p0[1], self.p1[1]])
        self.h_ends.set_data([self.p0[0], self.p1[0]], [self.p0[1], self.p1[1]])
        mid = 0.5 * (self.p0 + self.p1)
        self.h_mid.set_data([mid[0]], [mid[1]])

        # --- profiles
        for lab, ln in self.prof_lines.items():
            ln.set_data(d, profs[lab])
        self.ax_prof.set_xlim(0, max(d[-1], 1e-6))
        ymax = max([np.nanmax(v) for v in profs.values()] + [1e-9])
        self.ax_prof.set_ylabel("Mass-normalised [%]" if self.norm_mode == "norm"
                                else "Intensity [counts]")
 
        if self.log_scale:
            # log axes cannot show <= 0, so find the smallest positive value
            pos = np.concatenate([v[np.isfinite(v) & (v > 0)]
                                  for v in profs.values()] + [np.array([])])
            ymin = np.percentile(pos, 1) if pos.size else 1e-3
            self.ax_prof.set_yscale("log")
            self.ax_prof.set_ylim(max(ymin * 0.5, 1e-12), ymax * 25.0)
            self.ax_prof.yaxis.set_major_locator(mticker.FixedLocator([0.1, 1, 10, 100]))
        else:
            self.ax_prof.set_yscale("linear")
            if self.norm_mode == "norm":
                self.ax_prof.set_ylim(-5, 150)
                self.ax_prof.yaxis.set_major_locator(mticker.FixedLocator(np.arange(0, 101, 25)))
                self.ax_prof.yaxis.set_minor_locator(mticker.MultipleLocator(5))
            else:
                self.ax_prof.set_ylim(0, 1.4 * ymax)
        self.ax_prof.set_xlabel(f"Distance ({self.units})")

        # --- band-averaged spectrum
        if self.ax_spec is not None:
            mask = band_mask((self.ny, self.nx), self.p0, self.p1, self.width)
            if mask.any():
                spec = self.si[mask].mean(axis=0)
                self.spec_line.set_data(self.energy, spec)
                if self.si_bg is not None:
                    self.bg_line.set_data(self.energy, self.si_bg[mask].mean(axis=0))
                self.ax_spec.set_xlim(self.energy[0], self.energy[-1])
                self.ax_spec.set_ylim(0, 1.15 * max(spec.max(), 1e-9))
                self.ax_spec.set_title(f"mean spectrum of band  ({mask.sum()} px)",
                                       fontsize=9, loc="left")

        # --- text boxes / info
        for name, val in zip(("x0", "y0", "x1", "y1"),
                             (self.p0[0], self.p0[1], self.p1[0], self.p1[1])):
            self.tb_axes[name].set_val(f"{val:.1f}")   # does not fire on_submit

        self.txt_info.set_text(
            f"length : {L:7.1f} px\n"
            f"         {L * self.pixel_size:7.3f} {self.units}\n"
            f"width  : {self.width:7.1f} px\n"
            f"points : {len(d):7d}\n"
            f"step   : {(d[1] - d[0]) if len(d) > 1 else 0:7.4f} {self.units}"
        )
        self.fig.canvas.draw_idle()

    # ---------------------------------------------------------------- export
    def save_csv(self, path=None):
        t_px, profs, adf_p = self.compute()
        d = t_px * self.pixel_size
        path = path or self._next_name("linescan", "csv")
        header = [f"distance_{self.units}", "x_px", "y_px", "ADF"] + list(profs)
        v = self.p1 - self.p0
        L = np.hypot(*v)
        u = v / L if L > 1e-9 else np.zeros(2)
        xs, ys = self.p0[0] + u[0] * t_px, self.p0[1] + u[1] * t_px
        data = np.column_stack([d, xs, ys, adf_p] + [profs[k] for k in profs])
        meta = (f"p0=({self.p0[0]:.2f},{self.p0[1]:.2f}) "
                f"p1=({self.p1[0]:.2f},{self.p1[1]:.2f}) "
                f"width_px={self.width:g} smooth={self.smooth} "
                f"mode={self.norm_mode} pixel_size={self.pixel_size}{self.units}")
        np.savetxt(path, data, delimiter=",", header=meta + "\n" + ",".join(header),
                   comments="# ", fmt="%.6g")
        print(f"[saved] {path}")

    def save_png(self, path=None):
        path = path or self._next_name("linescan", "png")
        self.fig.savefig(path, dpi=300, facecolor="white")
        print(f"[saved] {path}")

    @staticmethod
    def _next_name(stem, ext):
        i = 1
        while os.path.exists(f"{stem}_{i:03d}.{ext}"):
            i += 1
        return f"{stem}_{i:03d}.{ext}"

    def show(self):
        plt.show()




# --------------------------------------------------------------------------
def load_with_hyperspy(si_path, adf_path=None):
    import hyperspy.api as hs

    def pick(objs, want_spectrum):
        objs = objs if isinstance(objs, list) else [objs]
        for o in objs:
            nsig = len(o.axes_manager.signal_axes)
            if want_spectrum and nsig == 1 and len(o.axes_manager.navigation_axes) == 2:
                return o
            if not want_spectrum and nsig == 2 and len(o.axes_manager.navigation_axes) == 0:
                return o
        return None

    si_obj = pick(hs.load(si_path, lazy=False), True)
    if si_obj is None:
        raise ValueError(f"No 2D-navigation spectrum image found in {si_path}")

    si = np.asarray(si_obj.data, dtype=float)          # (ny, nx, nE)
    eax = si_obj.axes_manager.signal_axes[0]
    energy = eax.offset + eax.scale * np.arange(si.shape[-1])
    if str(getattr(eax, "units", "keV")).lower() in ("ev", "electron volt"):
        energy = energy / 1000.0

    nav = si_obj.axes_manager.navigation_axes[0]
    pixel_size = float(nav.scale)
    units = str(getattr(nav, "units", "px"))
    if units in ("<undefined>", "", "None"):
        pixel_size, units = 1.0, "px"

    adf = None
    if adf_path:
        a = pick(hs.load(adf_path, lazy=False), False)
        if a is not None:
            adf = np.asarray(a.data, float)
    if adf is None:
        objs = hs.load(si_path, lazy=False)
        a = pick(objs, False)
        adf = np.asarray(a.data, float) if a is not None else si.sum(axis=2)
    if adf.shape != si.shape[:2]:
        adf = si.sum(axis=2)

    return si, energy, adf, pixel_size, units


def load_numpy(si_path, adf_path, e_offset, e_scale):
    si = np.load(si_path)
    if si.ndim != 3:
        raise ValueError(f"SI must be 3D (ny, nx, nE); got {si.shape}")
    si = si.astype(float)
    energy = e_offset + e_scale * np.arange(si.shape[-1])
    adf = np.load(adf_path).astype(float) if adf_path else si.sum(axis=2)
    return si, energy, adf



def launch(si=None, adf=None, elements=("Ni", "Cu"), lines=None, window=0.16,
           subtract_bg=True, pixel_size=None, units=None,
           e_offset=0.0, e_scale=0.01, width=5.0, title=None,
           window_frac=0.05, bg_percentile=5, bg_fast=True,
           poisson_correct=True, span_family=False, verbose=True):
    """Build the elemental maps and open the interactive tool.
 
    si   : path (hyperspy-readable or .npy), or an (ny, nx, nE) array
    adf  : path, or an (ny, nx) array, or None (uses SI total counts)
    subtract_bg : strip a rolling low-percentile background from the SI
        before integrating the windows.  See `rolling_background` for the
        meaning of window_frac / bg_percentile / bg_fast / poisson_correct.
    Keep the returned object in a variable or the widgets stop responding.
    """
   
    if isinstance(si, np.ndarray):
        si_arr = np.asarray(si, float)
        energy = e_offset + e_scale * np.arange(si_arr.shape[-1])
        adf_arr = si_arr.sum(axis=2) if adf is None else np.asarray(adf, float)
        pxs, un = 1.0, "px"
        title = title or "EDS analysis line scan"
    elif si is not None:
        if os.path.splitext(str(si))[1].lower() == ".npy":
            si_arr, energy, adf_arr = load_numpy(si, adf, e_offset, e_scale)
            pxs, un = 1.0, "px"
        else:
            si_arr, energy, adf_arr, pxs, un = load_with_hyperspy(si, adf)
        title = title or os.path.basename(str(si)) + "  EDS line scan"
    else:
        raise ValueError("give si=<path or array>, or demo=True")
 
    if pixel_size is not None:
        pxs = pixel_size
    if units is not None:
        un = units
 
    print(f"SI      : {si_arr.shape}   energy {energy[0]:.3f} - {energy[-1]:.3f} keV")
    print(f"ADF     : {adf_arr.shape}")
    print(f"pixel   : {pxs} {un}")
 
    lines = list(lines) if lines else []
    lines += [None] * (len(elements) - len(lines))
 
    # ---- strip the rolling-percentile background from the SI, once, up front
    si_bg = None
    if subtract_bg:
        import time as _t
        t0 = _t.time()
        si_bg = rolling_background(si_arr, window_frac=window_frac,
                                   percentile=bg_percentile, fast=bg_fast,
                                   poisson_correct=poisson_correct)
        si_arr = np.clip(si_arr - si_bg, 0, None)
        if verbose:
            print(f"background stripped (window_frac={window_frac}, "
                  f"p{bg_percentile}, fast={bg_fast}) in {_t.time() - t0:.1f} s")
 
    maps, info = {}, {}
    for el, ln in zip(elements, lines):
        m, name, e0, win = element_map(si_arr, energy, el, ln,
                                       width_kev=window,
                                       span_family=span_family, verbose=verbose)
        lab = f"{el}-{name}"
        maps[lab] = m
        info[lab] = {"energy": e0, "window": win}
        print(f"  map {lab:8s} @ {e0:6.3f} keV   window "
              f"{win[0]:.3f}-{win[1]:.3f} keV   max {m.max():.0f}")
 
    tool = LineScanTool(adf_arr, maps, si=si_arr, energy=energy, pixel_size=pxs,
                        units=un, title=title, line_info=info, init_width=width,
                        si_bg=si_bg)
    tool.show()
    return tool

In [31]:
#folder1  = "/Volumes/Swarnendu/07.30.26_AuBi/AuBi_CT_30_2/"
#folder2  = "/Volumes/Swarnendu/07.30.26_AuBi/AuBi_CT_5_1/"
#folder3  = "/Volumes/Swarnendu/07.30.26_AuBi/AuBi_CT_24_1/"
#folder4  = "/Volumes/Swarnendu/07.30.26_AuBi/AuBi_SO_30_1/"
#folder5  = "/Volumes/Swarnendu/07.30.26_AuBi/AuBi_SO_5_2/"
#folder6  = "/Volumes/Swarnendu/07.30.26_AuBi/AuBi_SO_24_1/"
#folder7  = "/Volumes/Swarnendu/07.30.26_AuBi/AuBi_CSO_30_1/"
#folder8  = "/Volumes/Swarnendu/07.30.26_AuBi/AuBi_CSO_5_2/"
folder9  = "/Volumes/Swarnendu/07.30.26_AuBi/AuBi_CSO_24_1/"

In [32]:
tool = launch(si= folder9 + "EDS Spectrum Image denoised.dm4", adf= folder9 + "ADF Image.dm4", 
              elements=["Au", "Bi"], window=0.24,
              span_family=True,      # widen the window to cover Ma AND Mb
              window_frac=0.05, bg_percentile=75,
              bg_fast=True, poisson_correct=True)

SI      : (216, 216, 1024)   energy -0.046 - 13.918 keV
ADF     : (216, 216)
pixel   : 0.001486065681092441 µm
background stripped (window_frac=0.05, p75, fast=True) in 6.5 s
    Au-Ma: window 2.002-2.234 keV (18 ch), max 24
  map Au-Ma    @  2.120 keV   window 2.002-2.234 keV   max 24
    Bi-Ma: window 2.302-2.643 keV (26 ch), max 1
  map Bi-Ma    @  2.419 keV   window 2.302-2.643 keV   max 1
